In [1]:
# with background genes
# ============================================================
# STEP 1. INSTALL PACKAGES
# ============================================================

!pip -q install gseapy pandas openpyxl

# ============================================================
# STEP 2. IMPORT
# ============================================================

from pathlib import Path
import time
import pandas as pd
import gseapy as gp

print("GSEApy version:", gp.__version__)


# ============================================================
# STEP 3. SETTINGS
# ============================================================

ADJ_P_CUTOFF = 0.05

TOP_R_VALUES = [10, 20, 30, 50, 70]

OUTPUT_DIR = Path("DLBCL_Enrichr_Sensitivity")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# STEP 4. YOUR RANKED GENE LIST
# ============================================================
# Ranking order is preserved.
# First 10 = Top-10
# First 20 = Top-20
# First 30 = Top-30
# First 50 = Top-50
# First 70 = Top-70

RANKED_GENES = [
    "AARS1",
    "ADA",
    "ADCY4",
    "ART1",
    "ATIC",
    "ATP5MC1",
    "BCL2A1",
    "BYSL",
    "CAD",
    "CCNB1",
    "CCNE1",
    "CCNF",
    "CCR1",
    "CCT3",
    "CCT5",
    "CDC20",
    "CDK1",
    "CDK2",
    "CDK4",
    "CDKN3",
    "CENPA",
    "CHAF1A",
    "CKAP5",
    "COX5A",
    "CTPS1",
    "CTSD",
    "DTYMK",
    "EBP",
    "EIF2S1",
    "EIF3B",
    "EIF4EP1",
    "ENO1",
    "ESA4",
    "ESPL1",
    "FCGR1A",
    "FIM3",
    "GCN1",
    "GINS1",
    "GLUL",
    "GM2A",
    "GPNMB",
    "GPS2",
    "H2AX",
    "HMBS",
    "HMGA1",
    "HSP90AB1",
    "HSPA5",
    "HSPD1",
    "IDH2",
    "IMPDH2",
    "KIF11",
    "KIF2C",
    "KPNA2",
    "LDHA",
    "LDHB",
    "LGALS3",
    "LOC105372824",
    "M6PR",
    "MCM2",
    "MCM3",
    "MCM5",
    "MCM6",
    "MCM7",
    "MIF",
    "MKI67",
    "MT1G",
    "MT2A",
    "MTHFD1",
    "MYBL2",
    "NME1"
]

assert len(RANKED_GENES) == 70

print("\nNumber of ranked genes:", len(RANKED_GENES))


# ============================================================
# STEP 5. GET CURRENT ENRICHR LIBRARIES
# ============================================================

print("\nRetrieving current Enrichr libraries...")

available_libraries = gp.get_library_name(
    organism="human"
)

available_libraries = sorted(set(available_libraries))

print(
    "Currently available libraries:",
    len(available_libraries)
)


# ============================================================
# STEP 6. PAPER-SPECIFIC LIBRARIES
# ============================================================

preferred_libraries = [

    "DisGeNET",

    "GO_Biological_Process_2025",
    "GO_Cellular_Component_2025",

    "Reactome_2022",

    "KEGG_2021_Human",

    "WikiPathway_2023_Human",

    "Jensen_TISSUES",

    "CCLE_Proteomics_2020",

    "NCI-60_Cancer_Cell_Lines"
]


selected_libraries = []


for library in preferred_libraries:

    if library in available_libraries:

        selected_libraries.append(library)

    else:

        print(
            "WARNING - currently unavailable:",
            library
        )


if len(selected_libraries) == 0:

    raise RuntimeError(
        "None of the requested Enrichr libraries are available."
    )


print("\n" + "=" * 70)
print("LIBRARIES USED IN EVERY EXPERIMENT")
print("=" * 70)

for library in selected_libraries:
    print("✓", library)


# Save exact library names

pd.DataFrame({
    "Library": selected_libraries
}).to_csv(
    OUTPUT_DIR / "libraries_used.csv",
    index=False
)


# ============================================================
# STEP 7. ENRICHR FUNCTION
# ============================================================

def run_enrichr(gene_list, top_r):

    """
    Run live Enrichr analysis.

    IMPORTANT:
    If Enrichr fails, the program raises an error.
    It does NOT convert failures into zero results.
    """

    print(
        f"\nSubmitting Top-{top_r} genes to Enrichr..."
    )

    try:

        enr = gp.enrichr(

            gene_list=gene_list,

            gene_sets=selected_libraries,

            # IMPORTANT: lowercase
            organism="human",

            # do not let GSEApy discard output
            outdir=None,

            # retrieve all returned results first
            cutoff=1.0,

            no_plot=True,

            verbose=False
        )

    except Exception as error:

        raise RuntimeError(
            f"\nEnrichr failed for Top-{top_r}.\n"
            f"Original error:\n{error}"
        )


    if enr.results is None:

        raise RuntimeError(
            f"Enrichr returned no result object for Top-{top_r}."
        )


    results = enr.results.copy()


    if results.empty:

        raise RuntimeError(
            f"Enrichr returned an empty table for Top-{top_r}."
        )


    # Convert statistical columns

    numeric_columns = [

        "P-value",
        "Adjusted P-value",
        "Old P-value",
        "Old Adjusted P-value",
        "Odds Ratio",
        "Combined Score"
    ]


    for column in numeric_columns:

        if column in results.columns:

            results[column] = pd.to_numeric(
                results[column],
                errors="coerce"
            )


    # Verify expected output columns

    required_columns = [
        "Gene_set",
        "Term",
        "Adjusted P-value"
    ]


    missing = [
        column
        for column in required_columns
        if column not in results.columns
    ]


    if missing:

        raise RuntimeError(
            f"Missing expected Enrichr columns: {missing}"
        )


    return results


# ============================================================
# STEP 8. SENSITIVITY ANALYSIS
# ============================================================

summary_rows = []

combined_significant_results = []

combined_all_results = []


for top_r in TOP_R_VALUES:

    print("\n")
    print("=" * 80)
    print(f"TOP {top_r} GENES")
    print("=" * 80)


    # --------------------------------------------
    # Select first R genes
    # --------------------------------------------

    genes = RANKED_GENES[:top_r]


    # Enrichr gene symbols should be unique

    genes = list(dict.fromkeys(genes))


    print(
        f"Genes submitted: {len(genes)}"
    )

    print("\n".join(genes))


    # --------------------------------------------
    # Create Top-R directory
    # --------------------------------------------

    top_dir = OUTPUT_DIR / f"Top_{top_r}"

    top_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    # Save exact input genes

    pd.DataFrame({
        "Rank": range(1, len(genes) + 1),
        "Gene": genes
    }).to_csv(
        top_dir / "input_genes.csv",
        index=False
    )


    # --------------------------------------------
    # LIVE ENRICHR ANALYSIS
    # --------------------------------------------

    results = run_enrichr(
        gene_list=genes,
        top_r=top_r
    )


    results["Top-R Genes"] = top_r


    # --------------------------------------------
    # Save ALL returned Enrichr results
    # --------------------------------------------

    results.to_csv(
        top_dir / "all_enrichr_results.csv",
        index=False
    )


    # --------------------------------------------
    # Remove exact duplicate library-term rows
    # --------------------------------------------

    results_unique = (
        results
        .drop_duplicates(
            subset=["Gene_set", "Term"]
        )
        .copy()
    )


    # --------------------------------------------
    # Significant terms
    # Adjusted P <= 0.05
    # --------------------------------------------

    significant = results_unique[
        results_unique["Adjusted P-value"]
        <= ADJ_P_CUTOFF
    ].copy()


    # Sort significant results

    significant = significant.sort_values(
        by=[
            "Adjusted P-value",
            "Combined Score"
        ],
        ascending=[
            True,
            False
        ]
    )


    significant.to_csv(
        top_dir / "significant_results_adjP_0.05.csv",
        index=False
    )


    # --------------------------------------------
    # COUNTS
    # --------------------------------------------

    total_returned_terms = len(
        results_unique
    )


    total_significant_terms = len(
        significant
    )


    # Libraries returning at least one result

    databases_returning_results = (
        results_unique["Gene_set"].nunique()
    )


    # Libraries with at least one significant result

    enriched_databases = (
        significant["Gene_set"].nunique()
        if not significant.empty
        else 0
    )


    # --------------------------------------------
    # SIGNIFICANT TERM RATE
    # --------------------------------------------

    significant_term_rate = (

        100.0
        * total_significant_terms
        / total_returned_terms

        if total_returned_terms > 0

        else 0.0
    )


    # --------------------------------------------
    # SUMMARY
    # --------------------------------------------

    summary_rows.append({

        "Top-R Genes":
            top_r,

        "Genes Submitted":
            len(genes),

        "Databases Returning Results":
            databases_returning_results,

        "No. of Enriched Databases":
            enriched_databases,

        "Total Returned Terms":
            total_returned_terms,

        "Significant Terms/Pathways (adj. p <= 0.05)":
            total_significant_terms,

        "Significant-Term Rate (%)":
            round(
                significant_term_rate,
                2
            )
    })


    combined_all_results.append(
        results_unique
    )

    combined_significant_results.append(
        significant
    )


    # --------------------------------------------
    # DISPLAY RESULT
    # --------------------------------------------

    print("\nACTUAL ENRICHR RESULT")

    print(
        "Databases returning results:",
        databases_returning_results
    )

    print(
        "Databases with significant enrichment:",
        enriched_databases
    )

    print(
        "Total returned terms:",
        total_returned_terms
    )

    print(
        "Significant terms (adj. p <= 0.05):",
        total_significant_terms
    )

    print(
        "Significant-term rate:",
        f"{significant_term_rate:.2f}%"
    )


    # --------------------------------------------
    # Show best significant terms
    # --------------------------------------------

    if not significant.empty:

        display_columns = [

            "Gene_set",
            "Term",
            "Adjusted P-value",
            "Combined Score"
        ]


        if "Genes" in significant.columns:
            display_columns.append("Genes")


        print(
            "\nTop significant enrichment results:"
        )

        display(
            significant[
                display_columns
            ].head(15)
        )

    else:

        print(
            "\nNo terms passed adjusted p <= 0.05."
        )


    # Avoid rapid repeated server requests

    time.sleep(3)


# ============================================================
# STEP 9. FINAL SENSITIVITY TABLE
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)


print("\n")
print("=" * 130)
print("FINAL QUANTITATIVE ENRICHMENT SENSITIVITY TABLE")
print("=" * 130)

display(summary_df)


# ============================================================
# STEP 10. SAVE SUMMARY CSV
# ============================================================

summary_df.to_csv(

    OUTPUT_DIR /
    "Enrichment_Sensitivity_Summary.csv",

    index=False
)


# ============================================================
# STEP 11. SAVE COMPLETE EXCEL WORKBOOK
# ============================================================

excel_file = (

    OUTPUT_DIR /
    "DLBCL_Enrichr_Sensitivity_Actual_Results.xlsx"

)


with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:


    summary_df.to_excel(
        writer,
        sheet_name="Sensitivity Summary",
        index=False
    )


    pd.DataFrame({
        "Library": selected_libraries
    }).to_excel(
        writer,
        sheet_name="Libraries Used",
        index=False
    )


    pd.DataFrame({
        "Rank": range(1, len(RANKED_GENES) + 1),
        "Gene": RANKED_GENES
    }).to_excel(
        writer,
        sheet_name="Ranked Genes",
        index=False
    )


    if combined_all_results:

        pd.concat(
            combined_all_results,
            ignore_index=True
        ).to_excel(
            writer,
            sheet_name="All Enrichr Results",
            index=False
        )


    if combined_significant_results:

        pd.concat(
            combined_significant_results,
            ignore_index=True
        ).to_excel(
            writer,
            sheet_name="Significant Results",
            index=False
        )


print("\n✓ ANALYSIS COMPLETED SUCCESSFULLY")

print(
    "\nExcel output:"
)

print(
    excel_file.resolve()
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.7/689.7 kB 9.1 MB/s eta 0:00:00
GSEApy version: 1.3.1

Number of ranked genes: 70

Retrieving current Enrichr libraries...
Currently available libraries: 228

LIBRARIES USED IN EVERY EXPERIMENT
✓ DisGeNET
✓ GO_Biological_Process_2025
✓ GO_Cellular_Component_2025
✓ Reactome_2022
✓ KEGG_2021_Human
✓ WikiPathway_2023_Human
✓ Jensen_TISSUES
✓ CCLE_Proteomics_2020
✓ NCI-60_Cancer_Cell_Lines


TOP 10 GENES
Genes submitted: 10
AARS1
ADA
ADCY4
ART1
ATIC
ATP5MC1
BCL2A1
BYSL
CAD
CCNB1

Submitting Top-10 genes to Enrichr...

ACTUAL ENRICHR RESULT
Databases returning results: 9
Databases with significant enrichment: 7
Total returned terms: 985
Significant terms (adj. p <= 0.05): 91
Significant-term rate: 9.24%

Top significant enrichment results:


,Gene_set,Term,Adjusted P-value,Combined Score,Genes
892,CCLE_Proteomics_2020,NCIH2286 LUNG TenPx19,0.000324,427.254176,CCNB1;ATIC;CAD;ADA;BYSL
721,WikiPathway_2023_Human,Purine Metabolism WP4792,0.000612,4973.589533,ATIC;ADA
722,WikiPathway_2023_Human,Purine Metabolism And Related Disorders WP4224,0.000991,2324.813163,ATIC;ADA
0,DisGeNET,Congenital hypoplastic anemia,0.001544,15739.234729,CAD;ADA
652,KEGG_2021_Human,Purine metabolism,0.002100,702.670364,ATIC;ADCY4;ADA
458,GO_Biological_Process_2025,Purine Ribonucleotide Biosynthetic Process (GO...,0.015415,976.581345,ATIC;ADCY4
1,DisGeNET,Aplasia Cutis Congenita,0.025326,394.278422,CCNB1;BCL2A1;CAD
459,GO_Biological_Process_2025,'De Novo' AMP Biosynthetic Process (GO:0044208),0.029925,2580.411794,ATIC
460,GO_Biological_Process_2025,Purine Ribonucleoside Metabolic Process (GO:00...,0.029925,2580.411794,ADA
461,GO_Biological_Process_2025,Adenosine Metabolic Process (GO:0046085),0.029925,2580.411794,ADA




TOP 20 GENES
Genes submitted: 20
AARS1
ADA
ADCY4
ART1
ATIC
ATP5MC1
BCL2A1
BYSL
CAD
CCNB1
CCNE1
CCNF
CCR1
CCT3
CCT5
CDC20
CDK1
CDK2
CDK4
CDKN3

Submitting Top-20 genes to Enrichr...

ACTUAL ENRICHR RESULT
Databases returning results: 9
Databases with significant enrichment: 8
Total returned terms: 1872
Significant terms (adj. p <= 0.05): 422
Significant-term rate: 22.54%

Top significant enrichment results:


,Gene_set,Term,Adjusted P-value,Combined Score,Genes
880,GO_Biological_Process_2025,G1/S Transition of Mitotic Cell Cycle (GO:0000...,2.426552e-11,5519.268434,CCNB1;CCNE1;CDK4;CCNF;CDK2;CDK1;CDKN3
881,GO_Biological_Process_2025,Cell Cycle G1/S Phase Transition (GO:0044843),2.426552e-11,5290.745658,CCNB1;CCNE1;CDK4;CCNF;CDK2;CDK1;CDKN3
1091,GO_Cellular_Component_2025,Cyclin-Dependent Protein Kinase Holoenzyme Com...,1.233939e-10,5442.195745,CCNB1;CCNE1;CDK4;CCNF;CDK2;CDK1
882,GO_Biological_Process_2025,Mitotic Cell Cycle Phase Transition (GO:0044772),7.810646e-10,2546.486246,CCNB1;CCNE1;CDK4;CCNF;CDK2;CDK1;CDKN3
1339,KEGG_2021_Human,Cell cycle,1.106274e-07,1451.943817,CDC20;CCNB1;CCNE1;CDK4;CDK2;CDK1
1340,KEGG_2021_Human,Oocyte meiosis,1.106274e-07,1376.034698,CDC20;CCNB1;CCNE1;CDK2;ADCY4;CDK1
1435,WikiPathway_2023_Human,Cell Cycle WP179,1.309489e-07,1518.015974,CDC20;CCNB1;CCNE1;CDK4;CDK2;CDK1
1436,WikiPathway_2023_Human,G1 To S Cell Cycle Control WP45,1.381511e-07,2168.892032,CCNB1;CCNE1;CDK4;CDK2;CDK1
1437,WikiPathway_2023_Human,DNA Damage Response WP707,1.381511e-07,1959.024947,CCNB1;CCNE1;CDK4;CDK2;CDK1
1438,WikiPathway_2023_Human,miRNA Regulation Of DNA Damage Response WP1530,1.381511e-07,1959.024947,CCNB1;CCNE1;CDK4;CDK2;CDK1




TOP 30 GENES
Genes submitted: 30
AARS1
ADA
ADCY4
ART1
ATIC
ATP5MC1
BCL2A1
BYSL
CAD
CCNB1
CCNE1
CCNF
CCR1
CCT3
CCT5
CDC20
CDK1
CDK2
CDK4
CDKN3
CENPA
CHAF1A
CKAP5
COX5A
CTPS1
CTSD
DTYMK
EBP
EIF2S1
EIF3B

Submitting Top-30 genes to Enrichr...

ACTUAL ENRICHR RESULT
Databases returning results: 9
Databases with significant enrichment: 8
Total returned terms: 2340
Significant terms (adj. p <= 0.05): 414
Significant-term rate: 17.69%

Top significant enrichment results:


...



TOP 70 GENES
Genes submitted: 70
AARS1
ADA
ADCY4
ART1
ATIC
ATP5MC1
BCL2A1
BYSL
CAD
CCNB1
CCNE1
CCNF
CCR1
CCT3
CCT5
CDC20
CDK1
CDK2
CDK4
CDKN3
CENPA
CHAF1A
CKAP5
COX5A
CTPS1
CTSD
DTYMK
EBP
EIF2S1
EIF3B
EIF4EP1
ENO1
ESA4
ESPL1
FCGR1A
FIM3
GCN1
GINS1
GLUL
GM2A
GPNMB
GPS2
H2AX
HMBS
HMGA1
HSP90AB1
HSPA5
HSPD1
IDH2
IMPDH2
KIF11
KIF2C
KPNA2
LDHA
LDHB
LGALS3
LOC105372824
M6PR
MCM2
MCM3
MCM5
MCM6
MCM7
MIF
MKI67
MT1G
MT2A
MTHFD1
MYBL2
NME1

Submitting Top-70 genes to Enrichr...

ACTUAL ENRICHR RESULT
Databases returning results: 9
Databases with significant enrichment: 8
Total returned terms: 4084
Significant terms (adj. p <= 0.05): 643
Significant-term rate: 15.74%

Top significant enrichment results:


,Gene_set,Term,Adjusted P-value,Combined Score,Genes
3453,Jensen_TISSUES,Lymph,6.817079e-18,717.940254,HSP90AB1;MCM7;ENO1;HSPD1;CDC20;LDHB;CCNB1;LDHA...
3281,WikiPathway_2023_Human,G1 To S Cell Cycle Control WP45,1.591964e-12,1940.130537,CCNB1;MCM7;CCNE1;CDK4;CDK2;MCM3;CDK1;MCM5;MCM6...
3280,WikiPathway_2023_Human,Cell Cycle WP179,1.591964e-12,1224.719041,CDC20;CCNB1;MCM7;ESPL1;CCNE1;CDK4;CDK2;MCM3;CD...
3454,Jensen_TISSUES,Ascites,1.802228e-12,346.447872,CCT3;HSP90AB1;MCM7;HSPA5;HMGA1;ENO1;MIF;HSPD1;...
3134,KEGG_2021_Human,Cell cycle,2.147028e-12,1166.035836,CDC20;CCNB1;MCM7;ESPL1;CCNE1;CDK4;CDK2;MCM3;CD...
2720,Reactome_2022,"Cell Cycle, Mitotic R-HSA-69278",6.677383e-11,392.198862,H2AX;GINS1;HSP90AB1;MCM7;CKAP5;CENPA;CDC20;CCN...
0,DisGeNET,Prostate carcinoma,2.201562e-10,191.797335,DTYMK;MCM7;BCL2A1;GPS2;ENO1;MKI67;COX5A;HSPD1;...
3455,Jensen_TISSUES,B-lymphoblastoid cell,4.666225e-10,143.109230,HSP90AB1;MCM7;CCNF;ENO1;KIF11;MKI67;CDC20;LGAL...
3456,Jensen_TISSUES,MOLT-4 cell,5.659163e-10,135.158797,HSP90AB1;MCM7;CCNF;ENO1;KIF11;MKI67;COX5A;HSPD...
3457,Jensen_TISSUES,Erythroid cell,5.659163e-10,133.711679,HSP90AB1;MCM7;CCNF;ENO1;KIF11;MKI67;COX5A;HSPD...




FINAL QUANTITATIVE ENRICHMENT SENSITIVITY TABLE


,Top-R Genes,Genes Submitted,Databases Returning Results,No. of Enriched Databases,Total Returned Terms,Significant Terms/Pathways (adj. p <= 0.05),Significant-Term Rate (%)
0,10,10,9,7,985,91,9.24
1,20,20,9,8,1872,422,22.54
2,30,30,9,8,2340,414,17.69
3,50,50,9,8,3456,458,13.25
4,70,70,9,8,4084,643,15.74



✓ ANALYSIS COMPLETED SUCCESSFULLY

Excel output:
/content/DLBCL_Enrichr_Sensitivity/DLBCL_Enrichr_Sensitivity_Actual_Results.xlsx
